In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import tensorflow as tf
from tensorflow import keras

In [2]:
X_user_train = pd.read_csv("../data/X_user_train.csv")
X_movie_train = pd.read_csv("../data/X_movie_train.csv")
y_train = pd.read_csv("../data/y_train.csv")

X_user_test = pd.read_csv("../data/X_user_test.csv")
X_movie_test = pd.read_csv("../data/X_movie_test.csv")
y_test = pd.read_csv("../data/y_test.csv")


In [3]:
#scale training data
X_user_unscaled = X_user_train.copy()
X_movie_unscaled = X_movie_train.copy()
y_unscaled = y_train.copy()

scaler_users = StandardScaler()
scaler_users.fit(X_user_unscaled)
X_user_train = scaler_users.transform(X_user_unscaled)
X_user_test = scaler_users.transform(X_user_test)

scaler_movies = StandardScaler()
scaler_movies.fit(X_movie_unscaled)
X_movie_train = scaler_movies.transform(X_movie_unscaled)
X_movie_test = scaler_movies.transform(X_movie_test)

scaler_y = MinMaxScaler((-1, 1))
y_unscaled = y_unscaled.values.reshape(-1, 1) # Reshape to 2D array for scaler to pass into fit and transfomr
scaler_y.fit(y_unscaled)
y_train = scaler_y.transform(y_unscaled)
y_test = scaler_y.transform(y_test.values.reshape(-1, 1))

#check inverse transform 
# user matix has NaN values
# first edition set NaN values to 0 (under test)
# improvement to test - set existence array with 1 or 0 to show existence (improves cold start but not useful for ranking)
print(np.allclose(X_user_unscaled, scaler_users.inverse_transform(X_user_train), equal_nan=True))
print(np.allclose(X_movie_unscaled, scaler_movies.inverse_transform(X_movie_train)))
print(np.allclose(y_unscaled, scaler_y.inverse_transform(y_train)))


True
True
True


In [ ]:
#set NaN values to 0
X_user_train[np.isnan(X_user_train)] = 0
X_user_test[np.isnan(X_user_test)] = 0


In [5]:
print("X_user_train[0]:", X_user_train[0], "\nX_movie_train[0]:", X_movie_train[0], "\ny_train[0]:", y_train[0])


X_user_train[0]: [-1.26852789 -2.33842713 -1.42965046 -1.32557236 -0.67418321  0.62007917
  0.5         1.6679979  -0.52008224  0.5        -0.4944423   0.5
 -0.22764365  1.391797    1.50784136 -2.50048509  0.18567752  0.41446902
 -0.88976695  0.5         0.5         0.5        -1.23413168  0.5
  1.47639604 -0.30518209 -0.24553128  0.72540696 -0.09209839  0.5
  0.5         0.5       ] 
X_movie_train[0]: [-6.70473015e-01 -5.67263386e-01 -2.64081915e-01 -2.98670105e-01
  1.36884644e+00 -4.52930709e-01 -1.09549218e-01  1.12296635e+00
  2.76880818e+00 -9.94184685e-02 -2.77991297e-01 -2.29955117e-01
 -1.89724946e-01 -3.01113173e-01  2.16197471e+00 -4.80490812e-01
  1.62920842e+00 -2.35334275e-01 -1.38936332e-01 -3.88866114e-01
 -2.58914483e-03 -5.17834172e-03 -3.25621858e-02 -8.04386345e-02
 -1.05498172e-01 -1.35513489e-01 -1.65757291e-01 -2.20724817e-01
 -3.63569430e-01  1.36382604e+00 -6.17211614e-01 -4.01784417e-01
 -8.99396998e-02] 
y_train[0]: [1.]


In [6]:
#Build initial NN

from tensorflow.keras.layers import Dense

num_outputs = 32

tf.random.set_seed(42)

userNN = tf.keras.models.Sequential([
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),
    Dense(num_outputs, activation='linear')
])

movieNN = tf.keras.models.Sequential([
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),
    Dense(num_outputs, activation='linear')
])

#create user input and point to base network
input_user = tf.keras.layers.Input(shape=(X_user_train.shape[1],))
vu = userNN(input_user)
#normalise to unit vector
vu = tf.keras.ops.normalize(vu, axis=1)


#create movie input and point to base network
input_movie = tf.keras.layers.Input(shape=(X_movie_train.shape[1],))
vm = movieNN(input_movie)
#normalise to unit vector
vm = tf.keras.ops.normalize(vm, axis=1)

#dot product layer
output = tf.keras.layers.Dot(axes=1)([vu, vm])

#model spec
model = tf.keras.Model(inputs=[input_user, input_movie], outputs=output)

model.summary()



Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 33)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 32)        │     45,472 │ input_layer[0][0] │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_1        │ (None, 32)        │     45,728 │ input_layer_2[0]… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalize           │ (None, 32)        │          0 │ sequential[0][0]  │
│ (Normalize)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalize_1         │ (None, 32)        │          0 │ sequential_1[0][… │
│ (Normalize)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot (Dot)           │ (None, 1)         │          0 │ normalize[0][0],  │
│                     │                   │            │ normalize_1[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 91,200 (356.25 KB)

 Trainable params: 91,200 (356.25 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
tf.random.set_seed(42)
cost_fn = tf.keras.losses.MeanSquaredError()
opt = keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=opt,loss=cost_fn)

In [8]:
tf.random.set_seed(42)
model.fit([X_user_train, X_movie_train], y_train, epochs=30)

Epoch 1/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - loss: 0.1471
Epoch 2/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 0.1410
Epoch 3/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 0.1381
Epoch 4/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 0.1355
Epoch 5/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 0.1333
Epoch 6/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 0.1319
Epoch 7/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 0.1301
Epoch 8/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - loss: 0.1290
Epoch 9/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 0.1283
Epoch 10/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 0.1277
Epoch 11/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 0.1271
Epoch 12/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 0.1266
Epoch 13/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 0.1261
Epoch 14/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 0.1255
Epoch 15/30
4662/4662 ━━━━━━━

In [9]:
#MSE eval on test set
model.evaluate([X_user_test, X_movie_test], y_test)

#test loss close to training loss, so model is not overfitting

##MSE eval on test set
model.evaluate([X_user_test, X_movie_test], y_test)

#test loss close to training loss, so model is not overfitting

#30 epochs and 1000 training samples
#0.15645070374011993 - only genre features
#0.1546485275030136 - genre and decade features

1166/1166 ━━━━━━━━━━━━━━━━━━━━ 1s 794us/step - loss: 0.1546
1166/1166 ━━━━━━━━━━━━━━━━━━━━ 1s 789us/step - loss: 0.1546


0.15456515550613403